<!-- WP5 延遲組：本機模型在 Colab L4 上一次跑一個模型（D175）。不需要任何 API key，也不呼叫 OpenAI。 -->
# Amidst WP5 延遲組（Colab L4，本機模型）

先讀 `docs/operations/WP5_COLAB_L4_README.md`。流程：確認 L4 → 解開從 Mac 帶來的包 → 固定 Ollama 版本與模型 digest（q4_K_M）→ **一次一個模型**暖機後跑題組 → 把結果打包帶回 Mac 的 `local/evidence/wp5/`。題目檔只含請求，沒有 Ground Truth；評分在 Mac。

In [ ]:
# 步驟 0：解開上傳的包、設定路徑並驗包。先把 wp5-pack_colab_l4_<日期>.zip 上傳到 /content（本筆記本旁邊）。
import glob, os, subprocess, sys, zipfile
from pathlib import Path
if not Path("tools/verify_pack.py").is_file():
    zips = sorted(glob.glob("wp5-pack_colab_l4_*.zip"))
    assert zips, "upload wp5-pack_colab_l4_<date>.zip next to this notebook first"
    zipfile.ZipFile(zips[-1]).extractall(".")
    os.chdir(zips[-1][:-4])
print("pack root:", os.getcwd())
sys.path.insert(0, str(Path("src").resolve()))
os.environ["PYTHONPATH"] = str(Path("src").resolve()) + os.pathsep + os.environ.get("PYTHONPATH", "")
v = subprocess.run([sys.executable, "tools/verify_pack.py", "."], capture_output=True, text=True)
print(v.stdout.strip()); assert v.returncode == 0 and "VALID" in v.stdout, v.stdout + v.stderr


In [ ]:
# 步驟 1：確認 GPU 是 NVIDIA L4；不是 L4 就停，因為非 L4 的延遲不能當正式結果。
import json, subprocess
from amidst.agent_eval.runner import detect_colab_gpu
proc = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True, check=False)
gpu = detect_colab_gpu(proc.stdout if proc.returncode == 0 else "")
print(json.dumps(gpu, ensure_ascii=False, indent=2))
assert gpu["is_l4"], "NOT an L4: do not report latency from this runtime"
GPU_LINE = proc.stdout.strip()

In [ ]:
# 步驟 2：設定。SET 是 pilot 或 formal（或 relaxed_exploration，要分開報告）。OLLAMA_VERSION 要在跑的當天釘死並寫進回傳紀錄。
SET = "pilot"
OLLAMA_VERSION = "AUTO_LATEST"   # 例如 0.x.y 可手動釘版；AUTO_LATEST 會在跑的當天取最新版並寫進 ENV.json
MODELS = ["qwen2.5:7b-instruct-q4_K_M", "llama3.1:8b-instruct-q4_K_M"]
QUESTIONS = "data/wp5_questions.jsonl"      # 從 Mac 帶來；不含 GT（verify_pack 會檢查）
OUT_ROOT = "RETURN/wp5"
SKIP_TYPES = "DP4"   # DP4 的即時呼叫要等 #532（ApiRanker）接上；接上後改成 ""
if OLLAMA_VERSION == "AUTO_LATEST":
    import json as _j, urllib.request as _u
    OLLAMA_VERSION = _j.loads(_u.urlopen("https://api.github.com/repos/ollama/ollama/releases/latest", timeout=20).read())["tag_name"].lstrip("v")
print("OLLAMA_VERSION pinned for this run:", OLLAMA_VERSION)

In [ ]:
# 步驟 3：安裝固定版本的 Ollama，並讓它一次只載入一個模型（延遲組一次只跑一個模型）。
import os, subprocess, time, urllib.request
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
subprocess.run(f"curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh", shell=True, check=True)
server = subprocess.Popen(["ollama", "serve"], stdout=open("ollama_serve.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
OLLAMA_VERSION_OUT = subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip()
print(OLLAMA_VERSION_OUT)

In [ ]:
# 步驟 4：拉模型、讀回 digest，寫 models.txt（name@digest）並驗證 q4_K_M 與 digest 鎖定；digest 不符就停。
import json, urllib.request
from pathlib import Path
from amidst.agent_eval.runner import verify_ollama_models
for m in MODELS:
    subprocess.run(["ollama", "pull", m], check=True)
tags = json.loads(urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=5).read())
digests = {x["name"]: x["digest"] for x in tags["models"]}
Path("models.txt").write_text("".join(f"{m}@{digests[m]}\n" for m in MODELS), encoding="utf-8")
check = verify_ollama_models(Path("models.txt").read_text(), tags)
print(json.dumps(check, ensure_ascii=False, indent=2))
assert check["valid"], check

In [ ]:
# 步驟 5：先 dry-run（0 次呼叫）核對題數與呼叫數。
import subprocess
r = subprocess.run(["python3", "-m", "amidst.agent_eval.wp5_runner", "--set", SET, "--arm", "colab_l4", "--skip-types", SKIP_TYPES, "--out", f"{OUT_ROOT}/dry_run"], capture_output=True, text=True)
print(r.stdout, r.stderr); assert r.returncode == 0
print(Path(f"{OUT_ROOT}/dry_run/estimate.json").read_text()[:1200])

In [ ]:
# 步驟 6：一次一個模型。每個模型先暖機一次（載入權重，不計入結果），再跑整個題組，最後卸載。
import json, subprocess, urllib.request
def ollama_chat(model, prompt):
    body = json.dumps({"model": model, "stream": False, "messages": [{"role": "user", "content": prompt}], "options": {"num_predict": 8}}).encode()
    urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:11434/api/chat", body, {"Content-Type": "application/json"}), timeout=120).read()
for m in MODELS:
    ollama_chat(m, "ping")  # warm-up, discarded
    out = f"{OUT_ROOT}/{SET}/{m.replace(':', '_').replace('/', '_')}"
    r = subprocess.run(["python3", "-m", "amidst.agent_eval.wp5_runner", "--set", SET, "--arm", "colab_l4", "--models", m, "--skip-types", SKIP_TYPES,
                        "--questions", QUESTIONS, "--live", "--out", out], capture_output=True, text=True)
    print(m, "rc=", r.returncode, r.stderr[-400:]); assert r.returncode == 0
    subprocess.run(["ollama", "stop", m], check=False)  # unload before the next model

In [ ]:
# 步驟 7：寫環境紀錄並打包，下載後放到 Mac 的 local/evidence/wp5/。
import hashlib, json, shutil
env = {"set": SET, "gpu": GPU_LINE, "ollama_version": OLLAMA_VERSION_OUT, "models_txt": Path("models.txt").read_text(),
       "digests": {m: digests[m] for m in MODELS}, "measure_latency": True, "serial_one_model_at_a_time": True}
Path(f"{OUT_ROOT}/ENV.json").write_text(json.dumps(env, ensure_ascii=False, indent=2), encoding="utf-8")
zip_path = shutil.make_archive(f"RETURN/wp5_{SET}_colab_l4", "zip", OUT_ROOT)
print(zip_path, hashlib.sha256(Path(zip_path).read_bytes()).hexdigest())
try:
    from google.colab import files; files.download(zip_path)
except ImportError:
    pass